# 06 — Evaluate + per-class best confidence

Two steps, both driven by `scripts/config.py`:

1. **Evaluate** the final (fine-tuned) checkpoint on the **independent TEST split**:
   per-class Precision/Recall + a PASS/FAIL report vs the targets in `config.TARGETS`.
2. **Per-class best confidence** — sweeps the confidence threshold on **test + validation
   combined** and records the best threshold **per class** (maximising F1, or recall under
   a precision floor — see `config.CONF_METRIC`). Results go to `best_conf.json` in the run
   folder **and** are appended to `runs/best_conf_log.csv` so they are easy to extract.

In [ ]:
import sys, json
sys.path.insert(0, '/home/jovyan/shared/s0598584/scripts')
import config as C
import train_pipeline as P

# Which run to evaluate: same naming as notebook 05.
RUN_NAME = f"{C.BASE_MODEL}{C.MODEL_SIZE}_{C.IMGSZ}_v1"
spec = P.make_spec(RUN_NAME, data=C.dataset_yaml(), imgsz=C.IMGSZ, model_size=C.MODEL_SIZE)
device, _ = P.resolve_devices()

# Use the fine-tuned best if present, else the coarse best.
ft = P.OUTPUT_ROOT/(RUN_NAME+'_ft')/'weights'/'best.pt'
co = P.OUTPUT_ROOT/RUN_NAME/'weights'/'best.pt'
final_best = ft if ft.exists() else co
assert final_best.exists(), f'no checkpoint found for {RUN_NAME} — train first (05)'
print('evaluating:', final_best)

report = P.evaluate(spec, final_best, device)
print(json.dumps(report['targets'], indent=2))

## Per-class best confidence (test + validation combined)

In [ ]:
conf_log = C.ROOT/'runs'/'best_conf_log.csv'
conf = P.per_class_best_conf(spec, final_best, device, conf_log)
print(json.dumps(conf['per_class'], indent=2))
print('\nappended to:', conf_log)

## Output artifact paths

In [ ]:
RUN_DIR = P.OUTPUT_ROOT/RUN_NAME
print('reports:')
for f in ['eval_report.json', 'eval_report.md', 'best_conf.json']:
    print(('  [ok] ' if (RUN_DIR/f).exists() else '  [--] ') + str(RUN_DIR/f))
print('\nglobal per-class confidence log:', C.ROOT/'runs'/'best_conf_log.csv')
outdir = P.OUTPUT_ROOT/(RUN_NAME+'_TEST')
print('\ntest-split plots in:', outdir)
for f in ['confusion_matrix.png','PR_curve.png','R_curve.png','F1_curve.png']:
    print(('  [ok] ' if (outdir/f).exists() else '  [--] ') + f)